<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/spam_email_example.png" align="center" width="30%">
</div>

<br>

# PRECISION, RECALL, AND THRESHOLD - APPLIED WITH SVM

<br>

**About:** An applied notebook demonstrating precision, recall, and threshold selection for spam email classification using a Support Vector Machine.

**Learning Goals:**
- Apply precision-recall analysis with an SVM classifier on a real-world text-derived dataset
- Identify how the scoring metric used during hyperparameter search shapes the resulting model
- Select a decision threshold to meet a target recall or precision requirement

**Keywords:** precision, recall, threshold, SVM, support vector machine, spam detection, binary classification, scikit-learn

**Prerequisite Knowledge:** (1) Python basics, (2) pandas and NumPy, (3) introductory supervised learning concepts, (4) `01_precision_recall_logistic_regression.ipynb` - for conceptual grounding on precision, recall, and the decision threshold

**Target User:** Self-learners who have completed the logistic regression notebook and want to apply the same precision-recall framework to a different model and domain

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: DATASET AND EXPLORATION](#Part_1)
> #### [PART 2: DATA PREPARATION](#Part_2)
> #### [PART 3: SVM PIPELINE AND HYPERPARAMETER TUNING](#Part_3)
> #### [PART 4: PRECISION-RECALL TRADE-OFF AND THRESHOLD ANALYSIS](#Part_4)

<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **DATASET** and Exploration

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/spam_email_example.png" align="center" width="35%" padding="10"><br>
    <br>
    Example spam email - the Spambase dataset captures word-frequency statistics from real spam and non-spam messages
</div>

#### CONTENTS:

> [PART 1.1: DATASET INFORMATION](#Part_1_1)<br>
> [PART 1.2: SANITY CHECK AND EXPLORATION](#Part_1_2)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: DATASET INFORMATION

<br>

We use the [Spambase dataset](https://archive.ics.uci.edu/ml/datasets/spambase) from the UCI Machine Learning Repository. The data was collected at Hewlett-Packard Labs and contains statistics derived from real email messages.

Key facts:
- **4,601 email samples**, each described by **57 numeric features** plus a binary label
- **Label**: `class` column - 1 = spam, 0 = not spam
- **Features** fall into three groups:
  - **48 word-frequency features** (`word_freq_*`): percentage of words in the email matching that word
  - **6 character-frequency features** (`char_freq_*`): percentage of characters matching specific punctuation
  - **3 capital-letter run features** (`capital_run_length_*`): statistics on sequences of consecutive capital letters
- **No missing values** in the published version

Spam detection is a precision-recall problem with a different cost asymmetry from cancer detection. A false negative (spam slips through) is annoying but low-cost. A false positive (a legitimate email is wrongly blocked) can cause the user to miss important messages - potentially more disruptive. This shifts the cost calculus toward higher precision compared to the cancer case.

___

**Sources Consulted:**
- [UCI Spambase Dataset](https://archive.ics.uci.edu/ml/datasets/spambase)

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: SANITY CHECK AND EXPLORATION

<br>

In [ ]:
## Load the spam dataset
df = pd.read_csv('SpamEmail.csv')
df.head(5)

In [ ]:
## Sanity check: shape, missing values, duplicates
print('Dataset shape:', df.shape)
print('\nMissing entries:')
print(df.isna().sum().sum(), 'total missing values')
print('\nDuplicated rows:', df.duplicated().sum())

In [ ]:
## Drop any duplicates before splitting
df = df[~df.duplicated()].reset_index(drop=True)
print('Shape after deduplication:', df.shape)

In [ ]:
## Column data types
df.dtypes.unique()

In [ ]:
## Class distribution
print('Class counts:')
print(df['class'].value_counts())
print(f"\nSpam fraction: {df['class'].mean():.1%}")

The dataset is moderately imbalanced at about 39% spam. As with the breast cancer dataset, this is not severe enough to require resampling, but it means accuracy alone would be a misleading metric - the non-spam majority class pulls accuracy up regardless of how well spam is detected.

In [ ]:
## Distribution of a word-frequency feature by class
fig, ax = plt.subplots(figsize=(8, 4))
for label, group in df.groupby('class')['word_freq_free']:
    ax.hist(group[group > 0], bins=20, alpha=0.6, label=f'class={label}')
ax.set_xlabel('word_freq_free (non-zero values)')
ax.set_ylabel('Count')
ax.set_title('Distribution of word_freq_free by Class (spam=1)')
ax.legend()
plt.tight_layout()
plt.show()

Words like "free" are strongly associated with spam in this dataset. Most non-spam emails have near-zero frequency for this word, while spam emails show a wider spread of occurrences. This is a common pattern in the feature space - many word-frequency features have near-zero values for non-spam but elevated values for spam.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Compute the mean value of `word_freq_make` for spam emails vs. non-spam emails. Based on the numbers, is this feature likely to be useful for detecting spam? Then pick one feature from the `capital_run_length_*` group and compare its mean across classes. Which group (`word_freq_*` or `capital_run_length_*`) shows a larger mean ratio between spam and non-spam?**

<br>

```python
### YOUR CODE HERE ###
word_means = ...
capital_means = ...

print(word_means)
print(capital_means)
```

<hr style="border: 2px solid#003262;" />

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **DATA** Preparation

#### CONTENTS:

> [PART 2.1: INPUT AND LABELS](#Part_2_1)<br>
> [PART 2.2: TRAINING AND TEST SETS](#Part_2_2)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: INPUT AND LABELS

<br>

All 57 feature columns are numeric and ready to use directly. The label column `class` is already 0/1 encoded - no `LabelEncoder` step is needed here, unlike in the breast cancer notebook where labels were strings.

In [ ]:
## Define features and labels
X = df.iloc[:, :-1]  # all columns except the last ('class')
y = df['class']      # 1 = spam, 0 = not spam

print('Feature matrix shape:', X.shape)
print('Label counts - 0 (not spam):', (y == 0).sum(), '| 1 (spam):', (y == 1).sum())

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: TRAINING AND TEST SETS

<br>

We use the same three-way split strategy as the logistic regression notebook: a held-out test set (15%) that is never used during model selection, a training set for fitting, and a validation set for hyperparameter tuning. The random state is fixed for reproducibility.

In [ ]:
## Hold out the test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=0
)

## Split training into a tuning-train and validation set
X_t, X_v, y_t, y_v = train_test_split(
    X_train, y_train, test_size=0.15, random_state=0
)

print('Train+Val size:', len(X_train), '| Test size:', len(X_test))
print('Tuning train:', len(X_t), '| Validation:', len(X_v))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The test set is used once, at the end, to report final performance. Explain in two sentences what specific problem would arise if you used the test set scores to choose between the recall-optimized and precision-optimized models (rather than making that choice based on the validation set or domain knowledge).**

<br>

```python
# No code required.
### YOUR ANSWER HERE ###
# ...
```

<hr style="border: 2px solid#003262;" />

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **SVM** Pipeline and Hyperparameter Tuning

#### CONTENTS:

> [PART 3.1: WHY SVM FOR SPAM DETECTION](#Part_3_1)<br>
> [PART 3.2: BUILDING THE PIPELINE](#Part_3_2)<br>
> [PART 3.3: HYPERPARAMETER TUNING](#Part_3_3)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: WHY SVM FOR SPAM DETECTION

<br>

A **Support Vector Machine (SVM)** finds the hyperplane in feature space that maximally separates two classes. The "support vectors" are the training examples closest to this hyperplane - the boundary is defined by these points alone, not the full dataset.

Key properties that make SVM appealing for spam detection:

<br>

**High-dimensional inputs** - the spam dataset has 57 features, all numeric. SVMs are well-suited to high-dimensional spaces where the number of features is large relative to the number of samples, because the decision boundary is a hyperplane rather than a piecewise function that can overfit complex surfaces.

<br>

**Kernel trick** - the `rbf` (radial basis function) kernel implicitly maps inputs into a higher-dimensional space where a linear boundary separates classes that were not linearly separable in the original feature space. This allows the SVM to capture nonlinear relationships between word frequencies and spam/non-spam labels without explicitly computing the high-dimensional mapping.

<br>

**Regularization via C** - the parameter `C` controls the trade-off between margin width and training error. Small `C` favors a wide margin with some misclassifications; large `C` allows a narrower margin to correctly classify more training points. This is the same regularization role that `C` plays in logistic regression.

___

**Note:** SVC with `probability=True` uses Platt scaling to produce calibrated probabilities. This adds a cross-validation step internally during `fit`, which increases training time. Without `probability=True`, `SVC` produces hard class predictions only and cannot support threshold moving. Verify the current behavior at [sklearn SVC docs](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html). `# TODO: verify probability calibration behavior for current sklearn version`

___

**Sources Consulted:**
- [scikit-learn SVM documentation](https://scikit-learn.org/stable/modules/svm.html)
- [sklearn SVC API](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: BUILDING THE PIPELINE

<br>

As in the logistic regression notebook, we combine scaling and the classifier into a `Pipeline`. Feature scaling is essential for SVMs: the RBF kernel computes distances between points in feature space, and features on different scales (e.g., `capital_run_length_total` can reach hundreds while word-frequency features are fractions) would dominate the distance calculation and bias the boundary.

We use `probability=True` so that `predict_proba` is available for threshold moving in Part 4.

In [ ]:
## Build the SVM pipeline
# TODO: verify gamma='auto' behavior in current sklearn version - 'auto' sets gamma = 1/n_features
# See: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', gamma='auto', probability=True, max_iter=int(1e7)))
])

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: HYPERPARAMETER TUNING

<br>

We tune `C` over a logarithmic range, using `GridSearchCV` with 4-fold stratified cross-validation. We run the search twice: once optimizing recall, once optimizing precision.

SVM training with `probability=True` is slower than logistic regression because of the internal Platt scaling step. On this dataset (3,000-4,000 training points, 57 features) each grid search may take a minute or more depending on hardware.

In [ ]:
## Tune C optimizing for recall
parameters = {'svc__C': np.logspace(0, 3, 10)}
split = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)

grid_recall = GridSearchCV(pipeline, parameters, cv=split, scoring='recall', n_jobs=-1)
grid_recall.fit(X_t, y_t)
pipeline_best_recall = grid_recall.best_estimator_

cv_recall = np.mean(cross_val_score(pipeline_best_recall, X_t, y_t, cv=split, scoring='recall'))
cv_prec = np.mean(cross_val_score(pipeline_best_recall, X_t, y_t, cv=split, scoring='precision'))

print('Best pipeline (optimized for recall)')
print(f'  C = {pipeline_best_recall.named_steps["svc"].C:.4f}')
print(f'  CV recall:    {cv_recall:.3f}')
print(f'  CV precision: {cv_prec:.3f}')

In [ ]:
## Tune C optimizing for precision
grid_prec = GridSearchCV(pipeline, parameters, cv=split, scoring='precision', n_jobs=-1)
grid_prec.fit(X_t, y_t)
pipeline_best_precision = grid_prec.best_estimator_

cv_recall_p = np.mean(cross_val_score(pipeline_best_precision, X_t, y_t, cv=split, scoring='recall'))
cv_prec_p = np.mean(cross_val_score(pipeline_best_precision, X_t, y_t, cv=split, scoring='precision'))

print('Best pipeline (optimized for precision)')
print(f'  C = {pipeline_best_precision.named_steps["svc"].C:.4f}')
print(f'  CV recall:    {cv_recall_p:.3f}')
print(f'  CV precision: {cv_prec_p:.3f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Compare the C values chosen by the recall-optimized and precision-optimized grid searches. Which selected a larger C? Explain in two sentences why a larger C might be preferred when optimizing for precision vs. recall, in terms of what a larger C does to the model's decision boundary.**

<br>

```python
c_recall = pipeline_best_recall.named_steps['svc'].C
c_precision = pipeline_best_precision.named_steps['svc'].C

### YOUR CODE HERE ###
print(f'C (recall-optimized):    {c_recall:.4f}')
print(f'C (precision-optimized): {c_precision:.4f}')

# Write your explanation as a comment:
# ...
```

<hr style="border: 2px solid#003262;" />

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **PRECISION-RECALL** Trade-off and Threshold Analysis

#### CONTENTS:

> [PART 4.1: THRESHOLD MOVING](#Part_4_1)<br>
> [PART 4.2: TESTING THE FINAL MODEL](#Part_4_2)<br>

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: THRESHOLD MOVING

<br>

We plot precision and recall as a function of the decision threshold for the recall-optimized SVM. The pattern is the same as in the logistic regression notebook: as the threshold rises, precision climbs and recall falls. The specific shape of the curve reflects the SVM's probability estimates for this dataset.

For spam detection, consider the consequences of each end of the curve:
- **Low threshold** - the model blocks most spam, but more legitimate emails get flagged. Users lose trust in email.
- **High threshold** - only high-confidence spam is blocked. Legitimate emails are almost never lost, but more spam gets through.

In [ ]:
## Fit recall-optimized SVM on tuning training set
pipeline_best_recall.fit(X_t, y_t)

## Sweep thresholds and compute precision and recall on the validation set
n = 20
thresh = np.linspace(0.01, 0.99, n)
precisions = np.zeros(n)
recalls = np.zeros(n)

for i, t in enumerate(thresh):
    y_pred = (pipeline_best_recall.predict_proba(X_v)[:, 1] >= t).astype(int)
    C = confusion_matrix(y_v, y_pred)
    tp, fp, fn = C[1, 1], C[0, 1], C[1, 0]
    precisions[i] = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recalls[i] = tp / (tp + fn) if (tp + fn) > 0 else 0.0

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh, precisions, 'r-o', markersize=4, label='Precision')
ax.plot(thresh, recalls, 'b-o', markersize=4, label='Recall')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall vs. Decision Threshold (SVM)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: TESTING THE FINAL MODEL

<br>

We choose a threshold that prioritizes precision for spam detection - users of a spam filter are more harmed by losing legitimate emails than by letting some spam through. The final model is retrained on the full training set before evaluation on the test set.

In [ ]:
## Retrain on full training set
pipeline_best_recall.fit(X_train, y_train)

## Choose a threshold - higher than 0.5 to favor precision for spam detection
thresh_chosen = 0.6

y_pred_test = (pipeline_best_recall.predict_proba(X_test)[:, 1] >= thresh_chosen).astype(int)
C_test = confusion_matrix(y_test, y_pred_test)

tp, fp, fn, tn = C_test[1, 1], C_test[0, 1], C_test[1, 0], C_test[0, 0]
test_recall = tp / (tp + fn)
test_precision = tp / (tp + fp)

print(f'Threshold: {thresh_chosen}')
print('Test set confusion matrix:')
print(C_test)
print(f'\nTest recall:    {test_recall:.3f}')
print(f'Test precision: {test_precision:.3f}')
print(f'\nFalse negatives (missed spam): {fn}')
print(f'False positives (blocked legitimate emails): {fp}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Compare the test-set precision and recall you obtained for the SVM spam detector to the test-set results from the breast cancer logistic regression notebook. Which model achieves higher recall? Which achieves higher precision? Based on the domain context of each problem (cancer screening vs. spam filtering), does the relative ranking of recall between the two models make sense? Explain in two sentences.**

<br>

```python
## Summarize results for comparison
## Fill in values from the logistic regression notebook
lr_recall = ...    # recall from 01_precision_recall_logistic_regression.ipynb
lr_precision = ... # precision from 01_precision_recall_logistic_regression.ipynb

svm_recall = test_recall
svm_precision = test_precision

print(f'{"Model":<10} {"Recall":>10} {"Precision":>12}')
print(f'{"LR (cancer)":<10} {lr_recall:>10.3f} {lr_precision:>12.3f}')
print(f'{"SVM (spam)":<10} {svm_recall:>10.3f} {svm_precision:>12.3f}')

# Your explanation:
# ...
```

<hr style="border: 2px solid#003262;" />

<hr style="border: 6px solid#003262;" />